# ⭐ Day 64: Hyperparameter Tuning Mastery
### *Efficient Optimization with Optuna*
---
**Day 64 of 369-day Python & AI Learning Path** 🐍🤖


## 🎯 Welcome to Day 64!
Welcome back, learner! You've built powerful ensembles, mastered cross-validation, and learned to diagnose your models. But here's the secret that separates good models from *great* ones: **hyperparameter tuning.**
Manually tweaking parameters is tedious, GridSearch is painfully slow, and RandomSearch wastes time on irrelevant regions of the search space. Today, we introduce **Optuna** — a cutting-edge, open-source hyperparameter optimization framework that uses **Bayesian optimization** and **early pruning** to find the best parameters in a fraction of the time.
Optuna is the go-to tool for Kaggle grandmasters and production ML engineers alike. It doesn't just search randomly; it learns from each trial, intelligently focusing on promising hyperparameter combinations and abandoning unpromising ones early. This means you'll achieve better performance with fewer computational resources — a game-changer for both experimentation and production pipelines.
By the end of this notebook, you'll be able to optimize any sklearn-compatible model, visualize the optimization landscape, understand which parameters matter most, and build reusable tuning functions that you can deploy on any project. Let's turn the art of tuning into a science! ⚙️🚀


## 📋 Table of Contents
1. [Why Hyperparameter Tuning Matters](#1-why-hyperparameter-tuning-matters)
2. [Traditional Methods vs Modern Optimization](#2-traditional-methods-vs-modern-optimization)
3. [Introduction to Optuna](#3-introduction-to-optuna)
4. [Basic Optuna Setup and Usage](#4-basic-optuna-setup-and-usage)
5. [Optimizing XGBoost / LightGBM with Optuna](#5-optimizing-xgboost--lightgbm-with-optuna)
6. [Optimizing Neural Networks with Optuna (Preview)](#6-optimizing-neural-networks-with-optuna-preview)
7. [Visualizing Optimization Results](#7-visualizing-optimization-results)
8. [Applying Optuna to Titanic Dataset](#8-applying-optuna-to-titanic-dataset)
9. [Applying Optuna to House Prices Dataset](#9-applying-optuna-to-house-prices-dataset)
10. [Best Practices for Hyperparameter Tuning](#10-best-practices-for-hyperparameter-tuning-in-real-projects)
11. [Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Best Practices](#-solutions--best-practices)


## 1. Why Hyperparameter Tuning Matters
Hyperparameters control the behavior of your learning algorithm — they are not learned from data but set before training. Poor choices lead to underfitting or overfitting; optimal choices unlock your model's true potential. 📈


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
warnings.filterwarnings('ignore')
from sklearn.datasets import make_classification, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score, classification_report
import xgboost as xgb
import lightgbm as lgb
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ All libraries imported successfully!")
print(f"Optuna version: {optuna.__version__}")


In [ ]:
# Demonstrate impact of hyperparameters on model performance
X_demo, y_demo = make_classification(n_samples=2000, n_features=20, n_informative=15, 
                                     n_redundant=5, random_state=42)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)
depths = [1, 3, 5, 10, 20, 30, None]
train_scores, test_scores = [], []
for d in depths:
    rf = RandomForestClassifier(max_depth=d, n_estimators=50, random_state=42)
    rf.fit(X_train_d, y_train_d)
    train_scores.append(accuracy_score(y_train_d, rf.predict(X_train_d)))
    test_scores.append(accuracy_score(y_test_d, rf.predict(X_test_d)))
plt.figure(figsize=(12, 6))
plt.plot([str(d) for d in depths], train_scores, 'o-', label='Training Accuracy', color='blue')
plt.plot([str(d) for d in depths], test_scores, 'o-', label='Test Accuracy', color='green')
plt.fill_between(range(len(depths)), train_scores, test_scores, alpha=0.2, color='red', label='Generalization Gap')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('📈 The Dramatic Impact of a Single Hyperparameter (max_depth)')
plt.legend()
plt.grid(True)
plt.show()
print("💡 The wrong max_depth can destroy generalization!")


## 2. Traditional Methods vs Modern Optimization
Let's compare Grid Search, Random Search, and Bayesian Optimization to understand *why* Optuna is superior. 🔍


In [ ]:
# Generate timing and performance comparison data
X_cmp, y_cmp = make_classification(n_samples=1500, n_features=15, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cmp, y_cmp, test_size=0.2, random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}
print("GridSearchCV combinations:", np.prod([len(v) for v in param_grid.values()]))
# Grid Search (limited to avoid timeout)
small_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10],
    'min_samples_split': [2, 5]
}
import time
start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42), small_grid, cv=3, n_jobs=-1, scoring='accuracy')
grid.fit(X_train_c, y_train_c)
grid_time = time.time() - start
print(f"GridSearchCV: {grid_time:.2f}s, Best Score: {grid.best_score_:.4f}")
# Random Search
start = time.time()
random = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_grid, n_iter=20, cv=3, 
                            n_jobs=-1, scoring='accuracy', random_state=42)
random.fit(X_train_c, y_train_c)
random_time = time.time() - start
print(f"RandomizedSearchCV: {random_time:.2f}s, Best Score: {random.best_score_:.4f}")


## 3. Introduction to Optuna
Optuna is a next-generation hyperparameter optimization framework with three core concepts: **Study**, **Trial**, and **Objective Function**. It supports **pruning** — stopping bad trials early to save time. 💡
- **Study**: An optimization session (a collection of trials)
- **Trial**: A single execution of the objective function with specific hyperparameters
- **Objective Function**: The function to minimize/maximize (e.g., validation loss)
- **Pruning**: Early stopping of unpromising trials using `optuna.pruners`


In [ ]:
# Core Optuna concepts demonstration
def objective_demo(trial):
    # Suggest hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    score = cross_val_score(model, X_train_c, y_train_c, cv=3, scoring='accuracy').mean()
    return score
# Create and run a study
study_demo = optuna.create_study(direction='maximize', study_name='rf_demo')
study_demo.optimize(objective_demo, n_trials=20, show_progress_bar=True)
print(f"Best trial: {study_demo.best_trial.number}")
print(f"Best value: {study_demo.best_value:.4f}")
print(f"Best params: {study_demo.best_params}")


## 4. Basic Optuna Setup and Usage
Let's build a clean, reusable Optuna workflow from scratch. This pattern works for any sklearn-compatible estimator. ⚙️


In [ ]:
# Reusable Optuna tuning function for any classifier
def tune_classifier(model_class, param_suggestions, X, y, n_trials=50, cv=5, metric='accuracy'):
    def objective(trial):
        params = {}
        for param_name, suggestion in param_suggestions.items():
            if suggestion['type'] == 'int':
                params[param_name] = trial.suggest_int(param_name, suggestion['low'], suggestion['high'])
            elif suggestion['type'] == 'float':
                params[param_name] = trial.suggest_float(param_name, suggestion['low'], suggestion['high'], log=suggestion.get('log', False))
            elif suggestion['type'] == 'categorical':
                params[param_name] = trial.suggest_categorical(param_name, suggestion['choices'])
        
        model = model_class(**params, random_state=42)
        score = cross_val_score(model, X, y, cv=cv, scoring=metric, n_jobs=-1).mean()
        return score
    
    study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study
# Example: Tune Random Forest
rf_params = {
    'n_estimators': {'type': 'int', 'low': 50, 'high': 500},
    'max_depth': {'type': 'int', 'low': 3, 'high': 30},
    'min_samples_split': {'type': 'int', 'low': 2, 'high': 20},
    'max_features': {'type': 'categorical', 'choices': ['sqrt', 'log2', None]}
}
print("🔍 Starting Random Forest optimization...")
rf_study = tune_classifier(RandomForestClassifier, rf_params, X_train_c, y_train_c, n_trials=30)
print(f"Best RF Accuracy: {rf_study.best_value:.4f}")
print(f"Best Params: {rf_study.best_params}")


## 5. Optimizing XGBoost / LightGBM with Optuna
XGBoost and LightGBM are competition staples. Their hyperparameter spaces are large, making them perfect candidates for Optuna's intelligent search. 📈


In [ ]:
# XGBoost optimization with Optuna
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    model = xgb.XGBClassifier(**params, random_state=42, n_jobs=-1)
    score = cross_val_score(model, X_train_c, y_train_c, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    return score
print("⚡ Optimizing XGBoost...")
study_xgb = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
print(f"Best XGBoost ROC-AUC: {study_xgb.best_value:.4f}")
print(f"Best XGBoost Params: {study_xgb.best_params}")


In [ ]:
# LightGBM optimization with Optuna
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1)
    score = cross_val_score(model, X_train_c, y_train_c, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    return score
print("🌿 Optimizing LightGBM...")
study_lgb = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)
print(f"Best LightGBM ROC-AUC: {study_lgb.best_value:.4f}")
print(f"Best LightGBM Params: {study_lgb.best_params}")


## 6. Optimizing Neural Networks with Optuna (Preview)
Optuna isn't limited to tree-based models. It can optimize neural network architectures, layer sizes, dropout rates, and even learning rate schedules. Here's a preview. 🧠


In [ ]:
# Neural Network hyperparameter optimization preview (using sklearn MLP as proxy)
from sklearn.neural_network import MLPClassifier
def objective_mlp(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    hidden_layers = []
    for i in range(n_layers):
        hidden_layers.append(trial.suggest_int(f'n_units_l{i}', 32, 256))
    
    params = {
        'hidden_layer_sizes': tuple(hidden_layers),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'max_iter': 500
    }
    model = MLPClassifier(**params, random_state=42, early_stopping=True)
    score = cross_val_score(model, X_train_c, y_train_c, cv=3, scoring='accuracy').mean()
    return score
print("🧠 Optimizing Neural Network architecture...")
study_mlp = optuna.create_study(direction='maximize')
study_mlp.optimize(objective_mlp, n_trials=20, show_progress_bar=True)
print(f"Best MLP Accuracy: {study_mlp.best_value:.4f}")
print(f"Best MLP Architecture: {study_mlp.best_params}")


## 7. Visualizing Optimization Results
Optuna provides beautiful built-in visualizations to understand your optimization landscape. Let's explore them! 📊


In [ ]:
# Optimization history
fig = optuna.visualization.plot_optimization_history(study_xgb)
fig.update_layout(title='📈 XGBoost Optimization History', width=900, height=500)
fig.show()


In [ ]:
# Parameter importance (which hyperparameters matter most?)
fig = optuna.visualization.plot_param_importances(study_xgb)
fig.update_layout(title='🔍 XGBoost Hyperparameter Importance', width=900, height=500)
fig.show()


In [ ]:
# Contour plot: interaction between two key parameters
fig = optuna.visualization.plot_contour(study_xgb, params=['max_depth', 'learning_rate'])
fig.update_layout(title='🗺️ Contour Plot: max_depth vs learning_rate', width=700, height=600)
fig.show()


In [ ]:
# Parallel coordinate plot: visualize all trials
fig = optuna.visualization.plot_parallel_coordinate(study_xgb)
fig.update_layout(title='🔄 Parallel Coordinate Plot: All Trials', width=1000, height=600)
fig.show()


## 8. Applying Optuna to Titanic Dataset (Classification)
Let's apply our Optuna skills to the Titanic survival prediction problem and compare tuned vs default performance. 🚢


In [ ]:
# Load and preprocess Titanic
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
titanic.dropna(inplace=True)
titanic['sex'] = LabelEncoder().fit_transform(titanic['sex'])
titanic['embarked'] = LabelEncoder().fit_transform(titanic['embarked'])
X_titanic = titanic.drop('survived', axis=1)
y_titanic = titanic['survived']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic
)
scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(X_train_t)
X_test_t = scaler_t.transform(X_test_t)
print(f"Titanic dataset ready: {X_titanic.shape}")


In [ ]:
# Default XGBoost performance
default_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
default_xgb.fit(X_train_t, y_train_t)
default_pred = default_xgb.predict(X_test_t)
default_acc = accuracy_score(y_test_t, default_pred)
print(f"Default XGBoost Accuracy: {default_acc:.4f}")
# Optuna-tuned XGBoost
def objective_titanic(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    model = xgb.XGBClassifier(**params, random_state=42)
    score = cross_val_score(model, X_train_t, y_train_t, cv=5, scoring='accuracy').mean()
    return score
study_titanic = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study_titanic.optimize(objective_titanic, n_trials=40, show_progress_bar=True)
print(f"\nTuned XGBoost Best CV Accuracy: {study_titanic.best_value:.4f}")
# Evaluate tuned model on test set
tuned_params = study_titanic.best_params.copy()
tuned_params.update({'use_label_encoder': False, 'eval_metric': 'logloss'})
tuned_xgb = xgb.XGBClassifier(**tuned_params, random_state=42)
tuned_xgb.fit(X_train_t, y_train_t)
tuned_pred = tuned_xgb.predict(X_test_t)
tuned_acc = accuracy_score(y_test_t, tuned_pred)
print(f"Tuned XGBoost Test Accuracy: {tuned_acc:.4f}")
print(f"Improvement: +{tuned_acc - default_acc:.4f}")


In [ ]:
# Visualize Titanic improvement
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# Bar comparison
models = ['Default XGBoost', 'Tuned XGBoost']
scores = [default_acc, tuned_acc]
colors = ['#3498db', '#e74c3c']
bars = axes[0].bar(models, scores, color=colors, edgecolor='black', alpha=0.8, width=0.5)
axes[0].set_ylim(0.75, 0.9)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('📊 Titanic: Default vs Tuned XGBoost')
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
                f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
# Optimization history for Titanic
history = [t.value for t in study_titanic.trials]
axes[1].plot(history, 'o-', color='purple', alpha=0.7)
axes[1].axhline(study_titanic.best_value, color='red', linestyle='--', label=f'Best: {study_titanic.best_value:.4f}')
axes[1].set_xlabel('Trial Number')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_title('📈 Optimization History (Titanic)')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()


## 9. Applying Optuna to House Prices Dataset (Regression)
Now let's optimize a regression model on the California Housing dataset. The process is nearly identical — only the metric changes! 🏠


In [ ]:
# Load California Housing
housing = fetch_california_housing()
X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
y_house = housing.target
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)
scaler_h = StandardScaler()
X_train_h = scaler_h.fit_transform(X_train_h)
X_test_h = scaler_h.transform(X_test_h)
print(f"California Housing ready: {X_house.shape}")


In [ ]:
# Default vs Tuned Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
default_gbr = GradientBoostingRegressor(random_state=42)
default_gbr.fit(X_train_h, y_train_h)
default_rmse = np.sqrt(mean_squared_error(y_test_h, default_gbr.predict(X_test_h)))
print(f"Default GBR RMSE: {default_rmse:.4f}")
# Optuna tuning for regression
def objective_house(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    # Negative RMSE for maximization
    scores = cross_val_score(model, X_train_h, y_train_h, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
    return scores.mean()
study_house = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
study_house.optimize(objective_house, n_trials=40, show_progress_bar=True)
print(f"\nTuned GBR Best CV Score: {study_house.best_value:.4f} (neg RMSE)")
# Evaluate tuned model
tuned_gbr = GradientBoostingRegressor(**study_house.best_params, random_state=42)
tuned_gbr.fit(X_train_h, y_train_h)
tuned_rmse = np.sqrt(mean_squared_error(y_test_h, tuned_gbr.predict(X_test_h)))
print(f"Tuned GBR Test RMSE: {tuned_rmse:.4f}")
print(f"Improvement: {default_rmse - tuned_rmse:.4f} lower RMSE")


In [ ]:
# Regression comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# RMSE comparison
models_r = ['Default GBR', 'Tuned GBR']
rmse_scores = [default_rmse, tuned_rmse]
colors_r = ['#3498db', '#e74c3c']
bars = axes[0].bar(models_r, rmse_scores, color=colors_r, edgecolor='black', alpha=0.8, width=0.5)
axes[0].set_ylabel('RMSE (lower is better)')
axes[0].set_title('📉 House Prices: Default vs Tuned GBR')
for bar, score in zip(bars, rmse_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
# Actual vs Predicted for tuned model
pred_tuned = tuned_gbr.predict(X_test_h)
axes[1].scatter(y_test_h, pred_tuned, alpha=0.5, edgecolors='black', linewidth=0.5)
axes[1].plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values (Tuned)')
axes[1].set_title('🏠 Tuned GBR: Actual vs Predicted')
axes[1].legend()
plt.tight_layout()
plt.show()


## 10. Best Practices for Hyperparameter Tuning in Real Projects
Before you unleash Optuna on every model, here are battle-tested guidelines: 💡
- **Start with Domain Knowledge**: Set reasonable ranges based on literature and experience
- **Use Pruning**: Always enable `MedianPruner` or `HyperbandPruner` to save time
- **Fix Random Seeds**: Ensure reproducibility across runs
- **Log Everything**: Use `study.trials_dataframe()` to analyze all experiments
- **Don't Overfit the Validation Set**: Keep a final holdout test set untouched until the end
- **Consider Multi-Objective**: Optuna supports optimizing accuracy AND inference speed simultaneously
- **Parallelize**: Use `n_jobs` in cross_val_score and run multiple Optuna studies in parallel
- **Start Coarse, Then Refine**: Do a broad search first, then narrow around the best region


## 🛠️ Hands-On Exercises

### Exercise 1: Set Up a Basic Optuna Study
Create an Optuna study to maximize accuracy. Define an objective function that suggests `C` (0.1 to 10.0, log scale) for LogisticRegression and returns 5-fold CV accuracy.


### Exercise 2: Optimize Hyperparameters for XGBoost
Write an objective function for XGBoost that optimizes `max_depth`, `learning_rate`, `n_estimators`, and `subsample`. Run 25 trials with pruning enabled.


### Exercise 3: Use Pruning to Speed Up Optimization
Implement `optuna.pruners.HyperbandPruner` in your study. Add `trial.report()` and `trial.should_prune()` inside your objective function to enable early stopping of bad trials.


### Exercise 4: Visualize Optimization History
After running a study, plot the optimization history using `optuna.visualization.plot_optimization_history()`. Interpret the plot: does the score improve over time?


### Exercise 5: Visualize Parameter Importance
Use `optuna.visualization.plot_param_importances()` to identify which hyperparameters had the biggest impact on model performance for your XGBoost study.


### Exercise 6: Compare Optuna with GridSearchCV
Run GridSearchCV on the same parameter space as your Optuna study. Compare the time taken and the best score achieved. Which method wins?


### Exercise 7: Tune a Full Pipeline (Preprocessing + Model)
Create a pipeline with StandardScaler and RandomForestClassifier. Use Optuna to optimize both preprocessing parameters (e.g., `with_mean` for scaler) and model hyperparameters simultaneously.


### Exercise 8: Create a Reusable Tuning Function
Write a generic function `tune_model(model_class, param_space, X, y, n_trials=50)` that accepts any sklearn model class and parameter space dictionary, then returns the best study.


### Exercise 9: Multi-Objective Optimization (Preview)
Set up a multi-objective Optuna study that simultaneously maximizes accuracy and minimizes model complexity (e.g., number of estimators). Use `optuna.create_study(directions=['maximize', 'minimize'])`.


### Exercise 10: Save and Load Optuna Studies
Save your completed study to a SQLite database using `study_name` and `storage` parameters. Then load it back and retrieve the best trial. This is essential for long-running experiments.


## Solutions & Best Practices (Review After Attempting)

### Exercise 1 Solution
The key to a basic Optuna study is defining a clear objective function and choosing the correct optimization direction.
```python
def objective(trial):
    C = trial.suggest_float('C', 0.1, 10.0, log=True)
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    return cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)
```
**Real-world tip**: Always use `log=True` for parameters like `C`, `learning_rate`, or regularization strengths — their effects are logarithmic.

### Exercise 2 Solution
XGBoost's most impactful parameters are typically `learning_rate`, `max_depth`, and `n_estimators`. Start by optimizing these three before adding others.
**Real-world tip**: Set `n_estimators` high and use early stopping via `trial.report()` to let pruning handle the rest.

### Exercise 3 Solution
Pruning requires reporting intermediate values. For tree-based models, you can report CV scores after each fold or use the model's built-in evaluation metric at each boosting round.
```python
if trial.should_prune():
    raise optuna.TrialPruned()
```
**Real-world tip**: `HyperbandPruner` is more aggressive than `MedianPruner` — use it when you have many fast trials and want aggressive pruning.

### Exercise 4 Solution
A good optimization history shows rapid improvement in early trials, then plateaus. If it's flat, your search space might be too narrow or your model might be insensitive to the parameters you're tuning.
**Real-world tip**: If the history is noisy, increase `n_trials` or reduce the search space around promising regions.

### Exercise 5 Solution
Parameter importance helps you focus your tuning efforts. If `learning_rate` dominates, you can fix other parameters and do a fine-grained search on just that one.
**Real-world tip**: Use this insight to build staged tuning pipelines — first tune the most important parameters, then the secondary ones.

### Exercise 6 Solution
GridSearchCV will almost always take longer and may find worse results than Optuna on large search spaces. Optuna shines when you have >10 hyperparameters or continuous ranges.
**Real-world tip**: Use GridSearch only for very small spaces (<100 combinations) or when you need guaranteed coverage of specific values.

### Exercise 7 Solution
Pipelines can be optimized by suggesting parameters for each step. Use step names as prefixes (e.g., `model__max_depth`) if using sklearn's Pipeline with Optuna.
**Real-world tip**: Preprocessing parameters usually have less impact than model parameters — optimize them last.

### Exercise 8 Solution
A reusable function should accept a model class, a parameter specification dict with types/ranges, and return both the study and the best model instance.
**Real-world tip**: Save this function in a `utils.py` file and import it across all your projects.

### Exercise 9 Solution
Multi-objective optimization returns a Pareto front of solutions. You can choose the model that best balances your competing objectives (e.g., accuracy vs speed).
**Real-world tip**: This is incredibly useful for production where you must balance performance with inference latency.

### Exercise 10 Solution
Persistent storage ensures you never lose optimization progress. SQLite is perfect for local development; PostgreSQL for team environments.
```python
study = optuna.create_study(study_name='my_study', storage='sqlite:///optuna.db', load_if_exists=True)
```
**Real-world tip**: Use `optuna-dashboard` to visually monitor running studies in real-time.


---
## 🎓 Day 64 Complete!
You now have modern hyperparameter tuning skills that can significantly boost model performance. You've moved beyond brute-force grid search to intelligent, efficient Bayesian optimization. You can diagnose which parameters matter, visualize optimization landscapes, and build reusable tuning pipelines that save hours of manual experimentation.
### 🔮 Coming Up Tomorrow: Day 65
**Model Interpretability with SHAP Deep Dive** — Learn how to explain any model's predictions with SHAP values. We'll demystify black-box models, generate force plots and summary plots, and ensure your high-performing models are also trustworthy and explainable! 🚀
---
*Python & AI Learning Path | Day 64 / 369* ⭐
